# 🌿 Plant Disease Classification — Competition Pipeline

**Competition:** SmellsLikeAISpirit Battle Royale  
**Dataset:** PlantVillage (39 classes, ~43.7k train images, ~11k test images)  
**Model:** EfficientNet-B3 (pretrained on ImageNet)  
**Strategy:** Two-phase fine-tuning + CutMix/MixUp + TTA + Fold Ensemble  

### Pipeline Overview
1. Install dependencies
2. Download datasets from HuggingFace
3. Prepare data & validate images
4. Train with 2-fold cross-validation (two-phase: freeze → unfreeze)
5. Run inference with Test-Time Augmentation (TTA)
6. Generate submission.csv

## Cell 1 — Install Dependencies
Install all required packages. `torch`, `torchvision`, `numpy`, `pandas`, `Pillow` are pre-installed on RunPod.

In [ ]:
!pip install -q timm ttach datasets huggingface_hub scikit-learn tqdm

## Cell 2 — Imports & Configuration
All imports in one place + the config class that controls every hyperparameter.

In [ ]:
import os
import sys
import json
import random
import subprocess
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm
import ttach as tta
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")


class CFG:
    """All hyperparameters and paths in one place."""
    
    # --- Paths ---
    data_dir = "./data"                    # Where datasets are cloned
    output_dir = "./outputs"               # Checkpoints + submission saved here

    # --- Model ---
    model_name = "efficientnet_b3"          # Architecture (also try: mobilenetv2_120d, tf_efficientnet_b4)
    pretrained = True                       # Use ImageNet pretrained weights
    num_classes = 39                        # Will be updated dynamically from data

    # --- Training ---
    img_size = 300                          # Input resolution (300 = EfficientNet-B3 native)
    batch_size = 64                         # Reduce to 32 if GPU has < 16GB VRAM
    num_workers = 4                         # DataLoader workers

    # Phase 1: Train classifier head only (backbone frozen)
    phase1_epochs = 5
    phase1_lr = 1e-3                        # Higher LR ok since only head is trainable

    # Phase 2: Unfreeze backbone, fine-tune everything
    phase2_epochs = 15                      # Max epochs (early stopping will kick in ~epoch 8-10)
    phase2_lr = 2e-5                        # Low LR to avoid destroying pretrained features
    weight_decay = 0.01                     # AdamW weight decay for regularization

    # --- Augmentation ---
    label_smoothing = 0.1                   # Prevents overconfident predictions
    cutmix_prob = 0.5                       # Probability of CutMix per batch (cuts & pastes image regions)
    cutmix_alpha = 1.0                      # Beta distribution param for CutMix
    mixup_prob = 0.3                        # Probability of MixUp per batch (blends images)
    mixup_alpha = 0.4                       # Beta distribution param for MixUp

    # --- Cross-Validation ---
    n_folds = 5                             # Total stratified folds
    train_folds = [0, 1]                    # Which folds to actually train (2 for speed)

    # --- Test-Time Augmentation ---
    use_tta = True                          # Flip + rotate at inference, average predictions

    # --- Other ---
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    mixed_precision = True                  # FP16 training for speed (2x faster on modern GPUs)
    early_stopping_patience = 7             # Stop if no improvement for N epochs


print(f"Device: {CFG.device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram/1e9:.1f} GB)")

## Cell 3 — Seed Everything
Set random seeds for reproducibility across Python, NumPy, and PyTorch.

In [ ]:
def seed_everything(seed):
    """Ensure reproducible results across runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True     # Speeds up training when input size is constant

seed_everything(CFG.seed)
os.makedirs(CFG.output_dir, exist_ok=True)
print("✅ Seeds set, output directory created")

## Cell 4 — Download Datasets
Clone the train and test datasets from HuggingFace using `git clone` + `git lfs`.  
This downloads ~2GB of images. Skips if already downloaded.

In [ ]:
def download_data():
    """Clone train + test datasets from HuggingFace via git."""
    data_dir = Path(CFG.data_dir)
    os.makedirs(data_dir, exist_ok=True)

    # Install git-lfs if not available (needed for large image files)
    try:
        subprocess.run(["git", "lfs", "version"], capture_output=True, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("📦 Installing git-lfs...")
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], capture_output=True)
        subprocess.run(["git", "lfs", "install"], check=True)

    train_repo = data_dir / "plant-disease-train"
    test_repo = data_dir / "plant-disease-test"

    if not train_repo.exists():
        print("📥 Cloning training dataset (may take a few minutes)...")
        subprocess.run([
            "git", "clone",
            "https://huggingface.co/datasets/SmellsLikeAISpirit/plant-disease-train",
            str(train_repo)
        ], check=True)
        print("✅ Training data downloaded")
    else:
        print("✅ Training data already exists")

    if not test_repo.exists():
        print("📥 Cloning test dataset...")
        subprocess.run([
            "git", "clone",
            "https://huggingface.co/datasets/SmellsLikeAISpirit/plant-disease-test",
            str(test_repo)
        ], check=True)
        print("✅ Test data downloaded")
    else:
        print("✅ Test data already exists")


download_data()

## Cell 5 — Prepare Training DataFrame
Scan the cloned repo, find all images and their labels.  
Uses `train_labels.csv` if available, otherwise scans class subdirectories.  
Also validates images and pulls Git LFS files if needed.

In [ ]:
def _print_distribution(df):
    """Print class distribution with warnings for minority classes."""
    print("\n📊 Class distribution:")
    dist = df["label"].value_counts()
    for cls, count in dist.items():
        marker = "⚠️" if count < 300 else "  "
        print(f"  {marker} {cls}: {count}")


def prepare_dataframe():
    """Build a DataFrame with filepath + label for every training image."""
    SKIP_DIRS = {".git", ".github", "__pycache__", ".ipynb_checkpoints", ".hf", ".cache"}
    base = Path(CFG.data_dir) / "plant-disease-train"

    # Show what's in the repo
    print(f"\n🔍 Scanning repo: {base}")
    if base.exists():
        top_items = sorted(base.iterdir())
        dirs = [x.name for x in top_items if x.is_dir() and x.name not in SKIP_DIRS]
        files = [x.name for x in top_items if x.is_file()]
        print(f"   Directories: {dirs[:15]}{'...' if len(dirs) > 15 else ''}")
        print(f"   Files: {files[:10]}")

    # Strategy 1: Use train_labels.csv (most reliable)
    for csv_path in [base / "train_labels.csv", base / "train" / "train_labels.csv"]:
        if csv_path.exists():
            print(f"   Found {csv_path}")
            labels_df = pd.read_csv(csv_path)
            print(f"   CSV columns: {list(labels_df.columns)}, rows: {len(labels_df)}")
            
            # Find where images actually live
            img_root = None
            sample_id = labels_df["id"].iloc[0]
            for candidate_root in [base / "train", base]:
                if not candidate_root.exists():
                    continue
                for sub in candidate_root.iterdir():
                    if sub.is_dir() and sub.name not in SKIP_DIRS:
                        if (sub / sample_id).exists():
                            img_root = candidate_root
                            break
                if img_root:
                    break
            
            if img_root:
                records = []
                for _, row in labels_df.iterrows():
                    fpath = img_root / row["label"] / row["id"]
                    if fpath.exists():
                        records.append({"filepath": str(fpath), "label": row["label"]})
                if records:
                    df = pd.DataFrame(records)
                    print(f"✅ Found {len(df)} training images via train_labels.csv")
                    _print_distribution(df)
                    return df

    # Strategy 2: Scan directories for class folders containing images
    candidates = [base / "train", base, Path(CFG.data_dir) / "train"]
    if base.exists():
        for child in sorted(base.iterdir()):
            if child.is_dir() and child.name not in SKIP_DIRS and child not in candidates:
                candidates.append(child)

    for d in candidates:
        if not d.exists():
            continue
        records = []
        class_count = 0
        for sub in sorted(d.iterdir()):
            if not sub.is_dir() or sub.name in SKIP_DIRS:
                continue
            imgs = [f for f in sub.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"} and f.is_file()]
            if imgs:
                class_count += 1
                for img_path in imgs:
                    records.append({"filepath": str(img_path), "label": sub.name})
        if class_count >= 10:
            df = pd.DataFrame(records)
            print(f"✅ Found {len(df)} training images across {class_count} classes in {d}")
            _print_distribution(df)
            return df

    raise FileNotFoundError("Cannot find training data!")


# Run it
df = prepare_dataframe()

## Cell 6 — Validate Images & Pull LFS
Check that image files are real images (not Git LFS pointers).  
If most files are LFS pointers, automatically runs `git lfs pull` to download them.

In [ ]:
def count_valid_images(df):
    """Check each image file — filter out LFS pointers and corrupted files."""
    valid_mask = []
    bad = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="  Checking"):
        try:
            fpath = row["filepath"]
            # LFS pointers start with 'version https://git-lfs'
            with open(fpath, "rb") as f:
                header = f.read(30)
            if header.startswith(b"version https://git-lfs"):
                valid_mask.append(False)
                bad += 1
                continue
            # Try opening the image to verify it's not corrupted
            img = Image.open(fpath)
            img.verify()
            valid_mask.append(True)
        except Exception:
            valid_mask.append(False)
            bad += 1
    return valid_mask, bad


print("🔍 Validating images...")
valid_mask, bad_count = count_valid_images(df)

# If >10% of files are bad, they're probably LFS pointers — pull the real data
if bad_count > len(df) * 0.1:
    print(f"\n⚠️  {bad_count}/{len(df)} files are invalid (likely Git LFS pointers)")
    print("   Running: git lfs pull (downloading ~2GB of images)...")
    repo_dir = Path(CFG.data_dir) / "plant-disease-train"
    subprocess.run(["git", "lfs", "pull"], cwd=str(repo_dir), check=True)
    print("✅ LFS pull complete. Re-validating...")
    valid_mask, bad_count = count_valid_images(df)

df = df[valid_mask].reset_index(drop=True)
print(f"✅ {len(df)} valid images ({bad_count} corrupted/skipped)")

if bad_count > 1000:
    print(f"❌ Still too many bad files! Run manually:")
    print(f"   cd {Path(CFG.data_dir) / 'plant-disease-train'} && git lfs pull")

## Cell 7 — Build Label Mapping
Create dictionaries to convert between class names and integer indices.  
Saves `label_mapping.json` for use during inference.

In [ ]:
def build_label_mapping(df):
    """Create bidirectional label <-> index mappings."""
    classes = sorted(df["label"].unique())
    label2idx = {label: idx for idx, label in enumerate(classes)}
    idx2label = {idx: label for label, idx in label2idx.items()}

    # Update num_classes dynamically from actual data
    CFG.num_classes = len(classes)
    print(f"🏷️  {len(classes)} classes mapped")

    # Save for inference later
    with open(Path(CFG.output_dir) / "label_mapping.json", "w") as f:
        json.dump({
            "label2idx": label2idx,
            "idx2label": {str(k): v for k, v in idx2label.items()},
        }, f, indent=2)

    return label2idx, idx2label


label2idx, idx2label = build_label_mapping(df)
print(f"Classes: {list(label2idx.keys())[:10]}...")

## Cell 8 — Create Stratified K-Fold Splits
Split data into 5 stratified folds (preserving class proportions in each fold).  
We'll only train folds 0 and 1 for speed, but each fold uses 80% train / 20% val.

In [ ]:
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(df, df["label"])):
    df.loc[val_idx, "fold"] = fold

print("📊 Fold distribution:")
for fold in range(CFG.n_folds):
    n = len(df[df["fold"] == fold])
    marker = "← will train" if fold in CFG.train_folds else ""
    print(f"   Fold {fold}: {n} val samples {marker}")

## Cell 9 — Data Augmentation Transforms
**Training transforms:** Random crop, flip, rotation, gentle color jitter, Gaussian blur, cutout.  
Color augmentation is kept mild to preserve disease-diagnostic color patterns (yellowing, brown spots, rust).  
**Validation transforms:** Just resize + normalize (deterministic).

In [ ]:
def get_train_transforms():
    """Training augmentations — aggressive geometry, gentle color."""
    return T.Compose([
        T.Resize((CFG.img_size + 32, CFG.img_size + 32)),   # Resize slightly larger
        T.RandomResizedCrop(CFG.img_size, scale=(0.8, 1.0)), # Random crop to target size
        T.RandomHorizontalFlip(p=0.5),                       # Leaf orientation doesn't matter
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(20),                                # ±20 degrees
        T.RandomAffine(degrees=0, translate=(0.1, 0.1)),     # Small shifts
        T.ColorJitter(brightness=0.1, contrast=0.1,          # GENTLE color changes
                      saturation=0.05, hue=0.02),            # (preserve disease colors!)
        T.RandomGrayscale(p=0.02),                           # Rarely convert to grayscale
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),     # Slight blur for robustness
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],              # ImageNet normalization
                    std=[0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.2, scale=(0.02, 0.15)),          # Cutout-style occlusion
    ])


def get_val_transforms():
    """Validation/test transforms — no randomness, just resize + normalize."""
    return T.Compose([
        T.Resize((CFG.img_size, CFG.img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])


print("✅ Transforms defined")

## Cell 10 — Dataset Classes
PyTorch Dataset wrappers for training and test images.  
Training dataset includes error handling — if an image is corrupted, it returns a random valid one instead of crashing.

In [ ]:
class PlantDiseaseDataset(Dataset):
    """Training/validation dataset. Loads images from filepaths in the DataFrame."""
    def __init__(self, df, label2idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["filepath"]).convert("RGB")
        except Exception:
            # If image is corrupted, return a random valid one instead of crashing
            return self.__getitem__(random.randint(0, len(self.df) - 1))
        label = self.label2idx[row["label"]]
        if self.transform:
            img = self.transform(img)
        return img, label


class PlantDiseaseTestDataset(Dataset):
    """Test dataset. Returns image + filename (no labels)."""
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fpath = self.file_list[idx]
        try:
            img = Image.open(fpath).convert("RGB")
        except Exception:
            img = Image.new("RGB", (CFG.img_size, CFG.img_size), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(fpath)


print("✅ Dataset classes defined")

## Cell 11 — CutMix & MixUp Augmentation
**CutMix:** Cuts a random rectangle from one image and pastes it onto another.  
Forces the model to recognize diseases from partial leaf views.  
**MixUp:** Blends two images together with a random weight.  
Both are applied during training only (not validation) and use a mixed loss function.

In [ ]:
def rand_bbox(size, lam):
    """Generate random bounding box for CutMix."""
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix_data(x, y, alpha=1.0):
    """Apply CutMix: cut a patch from one image, paste onto another."""
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    x1, y1, x2, y2 = rand_bbox(x.size(), lam)
    x[:, :, x1:x2, y1:y2] = x[index, :, x1:x2, y1:y2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (x.size(-1) * x.size(-2)))  # Adjust for actual area
    return x, y, y[index], lam


def mixup_data(x, y, alpha=0.4):
    """Apply MixUp: blend two images with random weight."""
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[index], y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Weighted loss for mixed samples: lam * loss(pred, y_a) + (1-lam) * loss(pred, y_b)"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print("✅ CutMix & MixUp defined")

## Cell 12 — Model Builder
Uses `timm` library to create an EfficientNet-B3 with ImageNet pretrained weights.  
The classifier head is automatically replaced with one matching our 39 classes.  
Includes functions to freeze/unfreeze the backbone for two-phase training.

In [ ]:
def build_model():
    """Create EfficientNet-B3 with pretrained weights and custom classifier."""
    model = timm.create_model(
        CFG.model_name, pretrained=CFG.pretrained,
        num_classes=CFG.num_classes,
        drop_rate=0.3,          # Dropout before classifier
        drop_path_rate=0.2,     # Stochastic depth (drops entire layers randomly)
    )
    total = sum(p.numel() for p in model.parameters())
    print(f"🧠 Model: {CFG.model_name} | Params: {total:,}")
    return model.to(CFG.device)


def freeze_backbone(model):
    """Freeze all layers except the classifier head (for Phase 1)."""
    for name, param in model.named_parameters():
        if "classifier" not in name and "fc" not in name and "head" not in name:
            param.requires_grad = False
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"❄️  Backbone frozen. Trainable: {t:,}")


def unfreeze_backbone(model):
    """Unfreeze all layers (for Phase 2 full fine-tuning)."""
    for param in model.parameters():
        param.requires_grad = True
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"🔥 Backbone unfrozen. Trainable: {t:,}")


print("✅ Model builder defined")

## Cell 13 — Training & Validation Functions
**train_one_epoch:** Runs one epoch with CutMix/MixUp, mixed precision, and gradient clipping.  
**validate:** Evaluates on validation set (no augmentation, no CutMix/MixUp).  
**get_sampler:** Creates WeightedRandomSampler to handle class imbalance.

In [ ]:
def get_sampler(df_fold):
    """WeightedRandomSampler: oversample minority classes so every class appears equally in batches."""
    class_counts = Counter(df_fold["label"].values)
    total = len(df_fold)
    weights = [total / (len(class_counts) * class_counts[df_fold.iloc[i]["label"]]) for i in range(len(df_fold))]
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch, phase="Phase 2"):
    """Train for one epoch. Applies CutMix/MixUp during Phase 2 only."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc=f"  {phase} | Epoch {epoch}")
    for images, labels in pbar:
        images = images.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)

        # Decide which augmentation to apply (only in Phase 2)
        do_cutmix = phase == "Phase 2" and np.random.random() < CFG.cutmix_prob
        do_mixup = phase == "Phase 2" and not do_cutmix and np.random.random() < CFG.mixup_prob

        optimizer.zero_grad()
        with autocast(enabled=CFG.mixed_precision):  # FP16 for speed
            if do_cutmix:
                images, ta, tb, lam = cutmix_data(images, labels, CFG.cutmix_alpha)
                loss = mixup_criterion(criterion, model(images), ta, tb, lam)
            elif do_mixup:
                images, ta, tb, lam = mixup_data(images, labels, CFG.mixup_alpha)
                loss = mixup_criterion(criterion, model(images), ta, tb, lam)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Prevent exploding gradients
        scaler.step(optimizer)
        scaler.update()

        # Track metrics
        running_loss += loss.item() * images.size(0)
        with torch.no_grad():
            if do_cutmix or do_mixup:
                outputs = model(images)
            _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{running_loss/total:.4f}", acc=f"{100.*correct/total:.1f}%")

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    """Evaluate model on validation set. No augmentation applied."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="  Validating", leave=False):
        images = images.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)
        with autocast(enabled=CFG.mixed_precision):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels


print("✅ Training & validation functions defined")

## Cell 14 — Train One Fold (Two-Phase Strategy)
**Phase 1 (5 epochs):** Freeze backbone, train only classifier head with lr=1e-3.  
The model learns to map pretrained features to our 39 plant disease classes.  

**Phase 2 (up to 15 epochs):** Unfreeze everything, fine-tune with lr=2e-5.  
Uses cosine annealing LR schedule, CutMix/MixUp, and early stopping.  
This is where accuracy goes from ~95% to 99%+.

In [ ]:
def train_fold(fold, df_train, df_val, label2idx, idx2label):
    """Train a single fold with two-phase strategy."""
    print(f"\n{'='*60}")
    print(f"  FOLD {fold}  |  Train: {len(df_train)}  |  Val: {len(df_val)}")
    print(f"{'='*60}")

    # Create datasets and dataloaders
    train_ds = PlantDiseaseDataset(df_train, label2idx, get_train_transforms())
    val_ds = PlantDiseaseDataset(df_val, label2idx, get_val_transforms())

    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, sampler=get_sampler(df_train),
        num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True,
    )

    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
    scaler = GradScaler(enabled=CFG.mixed_precision)
    best_acc = 0.0
    patience = 0

    # ---- PHASE 1: Classifier head only ----
    print(f"\n📌 Phase 1: Head only ({CFG.phase1_epochs} epochs, lr={CFG.phase1_lr})")
    freeze_backbone(model)
    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=CFG.phase1_lr, weight_decay=CFG.weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.phase1_epochs, eta_min=1e-6)

    for epoch in range(1, CFG.phase1_epochs + 1):
        train_one_epoch(model, train_loader, criterion, opt, scaler, epoch, "Phase 1")
        val_loss, val_acc, _, _ = validate(model, val_loader, criterion)
        sched.step()
        print(f"  → val_acc={val_acc:.4f}  val_loss={val_loss:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), Path(CFG.output_dir) / f"best_fold{fold}.pth")

    # ---- PHASE 2: Full fine-tuning ----
    print(f"\n📌 Phase 2: Full fine-tune ({CFG.phase2_epochs} epochs, lr={CFG.phase2_lr})")
    unfreeze_backbone(model)
    opt = optim.AdamW(model.parameters(), lr=CFG.phase2_lr, weight_decay=CFG.weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=CFG.phase2_epochs, T_mult=1, eta_min=1e-7)

    for epoch in range(1, CFG.phase2_epochs + 1):
        train_one_epoch(model, train_loader, criterion, opt, scaler, epoch, "Phase 2")
        val_loss, val_acc, preds, labels = validate(model, val_loader, criterion)
        sched.step(epoch)
        lr_now = opt.param_groups[0]["lr"]
        print(f"  → val_acc={val_acc:.4f}  val_loss={val_loss:.4f}  lr={lr_now:.2e}")

        if val_acc > best_acc:
            best_acc = val_acc
            patience = 0
            torch.save(model.state_dict(), Path(CFG.output_dir) / f"best_fold{fold}.pth")
            print(f"  ✅ New best! acc={val_acc:.4f}")
        else:
            patience += 1
            if patience >= CFG.early_stopping_patience:
                print(f"  ⏹️  Early stopping at epoch {epoch}")
                break

    print(f"\n🏆 Fold {fold} best: {best_acc:.4f}")

    # Show worst-performing classes
    model.load_state_dict(torch.load(Path(CFG.output_dir) / f"best_fold{fold}.pth"))
    _, _, preds, labels = validate(model, val_loader, criterion)
    report = classification_report(labels, preds,
        target_names=[idx2label[i] for i in range(CFG.num_classes)], output_dict=True)
    worst = sorted(
        [(n, m["f1-score"]) for n, m in report.items() if n in idx2label.values()],
        key=lambda x: x[1],
    )[:5]
    print("  ⚠️  5 worst classes:")
    for name, f1 in worst:
        print(f"     {name}: F1={f1:.3f}")

    del model
    torch.cuda.empty_cache()
    return best_acc


print("✅ train_fold function defined")

## Cell 15 — 🚀 Run Training
This is where the actual training happens. Trains each fold sequentially.  
Each fold takes ~15-20 minutes on RTX PRO 6000.

In [ ]:
fold_accs = []

for fold in CFG.train_folds:
    df_train = df[df["fold"] != fold].reset_index(drop=True)
    df_val = df[df["fold"] == fold].reset_index(drop=True)
    acc = train_fold(fold, df_train, df_val, label2idx, idx2label)
    fold_accs.append(acc)

print(f"\n{'='*60}")
print(f"  TRAINING COMPLETE")
print(f"  Fold accs: {[f'{a:.4f}' for a in fold_accs]}")
print(f"  Mean CV:   {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
print(f"{'='*60}")

## Cell 16 — Find Test Images
Locate all test images in the cloned repo.  
Also loads `sample_submission.csv` to ensure correct output format.

In [ ]:
def find_test_images():
    """Find all test images and sample_submission.csv."""
    base = Path(CFG.data_dir) / "plant-disease-test"
    test_images = []

    for d in [base / "test", base]:
        if not d.exists():
            continue
        for sub in ["images_0", "images_1"]:
            sub_dir = d / sub
            if sub_dir.exists():
                for ext in ["*.jpg", "*.JPG", "*.jpeg", "*.png"]:
                    test_images.extend(list(sub_dir.glob(ext)))
        if test_images:
            break
        for ext in ["*.jpg", "*.JPG", "*.jpeg", "*.png"]:
            test_images.extend(list(d.glob(ext)))
        if test_images:
            break

    # Fallback: recursive search
    if not test_images and base.exists():
        for ext in ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.png"]:
            for p in base.glob(ext):
                if ".git" not in str(p):
                    test_images.append(p)

    print(f"📸 Found {len(test_images)} test images")

    # Check for LFS pointers in test images
    if test_images:
        lfs_count = sum(1 for fp in test_images[:20]
                       if open(fp, 'rb').read(30).startswith(b'version https://git-lfs'))
        if lfs_count > 10:
            print("⚠️  Test images are LFS pointers. Pulling...")
            subprocess.run(["git", "lfs", "pull"], cwd=str(base), check=True)
            print("✅ Test LFS pull complete")

    # Load sample_submission.csv
    sample_df = None
    if base.exists():
        for sp in base.rglob("sample_submission.csv"):
            sample_df = pd.read_csv(sp)
            print(f"📋 Loaded {sp} ({len(sample_df)} rows)")
            break

    return test_images, sample_df


test_images, sample_df = find_test_images()

## Cell 17 — Run Inference with TTA
Load each fold's best model, run predictions on all test images.  
**TTA (Test-Time Augmentation):** For each image, predict on the original + flipped + rotated versions,  
then average the softmax probabilities. Typically adds 0.5-1.5% accuracy.

In [ ]:
@torch.no_grad()
def predict_with_tta(model, test_images):
    """Run inference with Test-Time Augmentation (flip + rotate, average predictions)."""
    if CFG.use_tta:
        transforms = tta.Compose([
            tta.HorizontalFlip(),
            tta.VerticalFlip(),
            tta.Rotate90(angles=[0, 90, 180, 270]),
        ])
        model = tta.ClassificationTTAWrapper(model, transforms, merge_mode="mean")

    model.eval()
    ds = PlantDiseaseTestDataset(test_images, get_val_transforms())
    loader = DataLoader(ds, batch_size=CFG.batch_size, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True)

    all_probs, all_fnames = [], []
    for images, fnames in tqdm(loader, desc="🔮 Predicting"):
        images = images.to(CFG.device, non_blocking=True)
        with autocast(enabled=CFG.mixed_precision):
            outputs = model(images)
        all_probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
        all_fnames.extend(fnames)

    return all_fnames, np.concatenate(all_probs, axis=0)


# Run inference with each fold model
all_fold_probs = []

for fold in CFG.train_folds:
    print(f"\n🔮 Inference with fold {fold}...")
    model = build_model()
    model.load_state_dict(torch.load(Path(CFG.output_dir) / f"best_fold{fold}.pth"))
    fnames, probs = predict_with_tta(model, test_images)
    all_fold_probs.append(probs)
    del model
    torch.cuda.empty_cache()

print(f"\n✅ Inference complete — {len(CFG.train_folds)} fold(s) predicted")

## Cell 18 — Generate submission.csv
Average predictions across all folds, map to class labels, and validate the output format.  
Must have exactly 10,976 rows, columns `id` and `label`, no duplicates.

In [ ]:
# Average softmax probabilities across folds (ensemble)
avg_probs = np.mean(all_fold_probs, axis=0)
predictions = np.argmax(avg_probs, axis=1)

# Build submission DataFrame
submission = pd.DataFrame({
    "id": fnames,
    "label": [idx2label[p] for p in predictions],
})

# Use sample_submission as template to guarantee correct IDs and order
if sample_df is not None:
    pred_map = dict(zip(submission["id"], submission["label"]))
    sample_df["label"] = sample_df["id"].map(pred_map)
    missing = sample_df["label"].isna().sum()
    if missing > 0:
        print(f"⚠️  {missing} test images not predicted — filling with 'other'")
        sample_df["label"] = sample_df["label"].fillna("other")
    submission = sample_df[["id", "label"]].copy()

# Validate format
expected_rows = 10976
print(f"\n✅ Validation:")
print(f"   Rows: {len(submission)} {'✅' if len(submission) == expected_rows else '❌ EXPECTED ' + str(expected_rows)}")
print(f"   Columns: {list(submission.columns)} {'✅' if list(submission.columns) == ['id', 'label'] else '❌'}")
print(f"   Duplicates: {submission['id'].duplicated().sum()} {'✅' if submission['id'].duplicated().sum() == 0 else '❌'}")
print(f"   Classes used: {submission['label'].nunique()}")

# Save
save_path = Path(CFG.output_dir) / "submission.csv"
submission.to_csv(save_path, index=False)
print(f"\n💾 Saved → {save_path}")

# Stats
print(f"\n📊 Top 10 predicted classes:")
print(submission["label"].value_counts().head(10).to_string())

max_probs = np.max(avg_probs, axis=1)
print(f"\n🎯 Confidence: mean={max_probs.mean():.3f}  min={max_probs.min():.3f}  low(<0.5)={int((max_probs<0.5).sum())}")

print(f"\n🎉 Done! Upload {save_path} to the competition page.")

## Cell 19 — Preview Some Predictions
Quick sanity check: show a few test image predictions with confidence scores.

In [ ]:
# Show 20 random predictions
sample = submission.sample(20, random_state=42)
print(f"{'ID':<25} {'Prediction':<50} {'Confidence':>10}")
print("─" * 85)

for _, row in sample.iterrows():
    # Find the index of this image in our predictions
    idx = fnames.index(row['id']) if row['id'] in fnames else None
    if idx is not None:
        conf = avg_probs[idx].max() * 100
        bar = "█" * int(conf / 5)
        print(f"{row['id']:<25} {row['label']:<50} {conf:>5.1f}% {bar}")
    else:
        print(f"{row['id']:<25} {row['label']:<50}     N/A")